In [1]:
from pyspark.sql import SparkSession

In [2]:
spark.catalog.listDatabases()

[Database(name='bootcamp', catalog='demo', description=None, locationUri='s3://warehouse/bootcamp')]

In [208]:
spark.catalog.listTables('bootcamp')

[Table(name='actors', catalog='demo', namespace=['bootcamp'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='match_details_bucketed', catalog='demo', namespace=['bootcamp'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='matches_bucketed', catalog='demo', namespace=['bootcamp'], description=None, tableType='MANAGED', isTemporary=False)]

In [220]:
%%sql
DROP TABLE IF EXISTS bootcamp.actor_films;

++
||
++
++

In [221]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.actor_films (
    actor STRING
    , actorid STRING
    , film STRING
    , year INT
    , votes INT
    , rating FLOAT
    , filmid STRING 
)
USING iceberg;

++
||
++
++

In [222]:
spark.catalog.listColumns('bootcamp.actor_films')

[Column(name='actor', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
 Column(name='actorid', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
 Column(name='film', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
 Column(name='year', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False),
 Column(name='votes', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False),
 Column(name='rating', description=None, dataType='float', nullable=True, isPartition=False, isBucket=False),
 Column(name='filmid', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False)]

In [223]:
schema = StructType([
    StructField("actor", StringType()),
    StructField("actorid", StringType()),
    StructField("film", StringType()),
    StructField("year", IntegerType()),
    StructField("votes", IntegerType()),
    StructField("rating", FloatType()),
    StructField("filmid", StringType())
])

In [224]:
df_actor_films = spark.read.format("jdbc")\
    .option("url","jdbc:postgresql://host.docker.internal:5432/")\
    .option("driver","org.postgresql.Driver")\
    .option("user","postgres")\
    .option("password","postgres")\
    .option("dbtable","actor_films") \
    .option("schema","schema") \
    .load()
df_actor_films.write.mode("append").saveAsTable("bootcamp.actor_films")

In [376]:
%%sql
DROP TABLE IF EXISTS bootcamp.actors;

++
||
++
++

In [377]:
%%sql
CREATE TABLE IF NOT EXISTS bootcamp.actors (
    actor STRING
    , actorid STRING
    , year INT
    , films array<struct<film: STRING, year: int, votes: int, rating: float, filmid: STRING>>
    , quality_class STRING
    , is_active BOOLEAN
)
USING iceberg;

++
||
++
++

In [348]:
spark.catalog.listColumns('bootcamp.actors')

[Column(name='actor', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
 Column(name='actorid', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
 Column(name='year', description=None, dataType='int', nullable=True, isPartition=False, isBucket=False),
 Column(name='films', description=None, dataType='array<struct<film:string,year:int,votes:int,rating:float,filmid:string>>', nullable=True, isPartition=False, isBucket=False),
 Column(name='quality_class', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
 Column(name='is_active', description=None, dataType='boolean', nullable=True, isPartition=False, isBucket=False)]

In [68]:
from pyspark.sql.types import *

In [ ]:
schema = StructType([
    StructField("actor", StringType()),
    StructField("actorid", IntegerType()),
    StructField("year", StringType()),
    StructField("films", StringType()),
    StructField("languagesSkills", ArrayType(StringType())),
])
df_actors = spark.read.format("jdbc")\
    .option("url","jdbc:postgresql://host.docker.internal:5432/")\
    .option("driver","org.postgresql.Driver")\
    .option("user","postgres")\
    .option("password","postgres")\
    .option("dbtable","actors").load()
df_actors.write.mode("overwrite").saveAsTable("bootcamp.actors")
df_actors.show()

In [228]:
%%sql
select * from bootcamp.actors

actor,actorid,year,films,quality_class,is_active


In [387]:
def do_actors_cumulative_transformation(spark, d_actors, d_actor_films, year, year1):
    query = f"""
        WITH last_year AS (
            SELECT actor, actorid, year, films, quality_class, is_active
            FROM actors
            WHERE year = '{year1}'       
        ), this_year AS (
            SELECT actor, actorid, year
            , array_agg(struct(
                         film
                        , year
                        , votes
                        , rating
                        , filmid)) AS films
            , ROUND(AVG(rating)) AS avg_rating
            FROM actor_films
            WHERE year = '{year}'
            GROUP BY actor, actorid, year
        )
        SELECT
            COALESCE(ls.actor, ts.actor) as actor
            , COALESCE(ls.actorid, ts.actorid) as actorid
            , COALESCE(ts.year, ls.year + 1) as year
            , case when ts.year is null then ls.films 
            when ls.year is not null then concat(ls.films, ts.films)
            when ls.year is null then ts.films 
            end AS films
            , CASE WHEN ts.year IS NOT NULL THEN
                    (CASE WHEN avg_rating > 8 THEN 'star'
                    WHEN avg_rating BETWEEN 7 AND 8 THEN 'good'
                    WHEN avg_rating BETWEEN 6 AND 7 THEN 'average'
                    ELSE 'bad' END)
                ELSE ls.quality_class
                END AS quality_class
            , ts.year IS NOT NULL AS is_active
            FROM last_year ls
            FULL OUTER JOIN this_year ts
            ON ls.actorid = ts.actorid
        ;    
    """
    d_actors.createOrReplaceTempView("actors")
    d_actor_films.createOrReplaceTempView("actor_films")
    return spark.sql(query)
#ARRAY(struct(film STRING, year INT, votes INT, rating STRING, filmid STRING))
#struct(film STRING, year STRING, votes STRING, rating STRING, filmid STRING)
#            , CASE WHEN ls.films IS NOT NULL THEN ls.films
#                ELSE array<struct<'',0,0,float('infinity'),''>> end 
#                || CASE WHEN ts.year IS NOT NULL THEN ts.films
#                    ELSE array<struct<'',0,0,float('infinity'),''>> END               AS films

In [388]:
def main():
    year = '1972'
    year1 = '1971'
    spark = SparkSession.builder \
      .master("local") \
      .appName("actors_cumulative") \
      .getOrCreate()
    
    output_df = do_actors_cumulative_transformation(spark, spark.table("bootcamp.actors"), spark.table("bootcamp.actor_films"), year, year1)
    #output_df.show()
    output_df.write.mode("append").insertInto("bootcamp.actors")

main()

24/12/08 03:39:28 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [389]:
%%sql
select * from bootcamp.actors where actor = 'Brigitte Bardot' --year = 1972 -- 

24/12/08 03:39:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


actor,actorid,year,films,quality_class,is_active
Brigitte Bardot,nm0000003,1971,"[Row(film='The Bear and the Doll', year=1970, votes=431, rating=6.400000095367432, filmid='tt0064779'), Row(film='Les novices', year=1970, votes=219, rating=5.099999904632568, filmid='tt0066164'), Row(film='Frenchie King', year=1971, votes=1054, rating=5.300000190734863, filmid='tt0067637'), Row(film='Rum Runners', year=1971, votes=469, rating=5.599999904632568, filmid='tt0066857')]",bad,True
Brigitte Bardot,nm0000003,1972,None,bad,False
Brigitte Bardot,nm0000003,1970,"[Row(film='The Bear and the Doll', year=1970, votes=431, rating=6.400000095367432, filmid='tt0064779'), Row(film='Les novices', year=1970, votes=219, rating=5.099999904632568, filmid='tt0066164')]",average,True
Brigitte Bardot,nm0000003,1972,"[Row(film='The Bear and the Doll', year=1970, votes=431, rating=6.400000095367432, filmid='tt0064779'), Row(film='Les novices', year=1970, votes=219, rating=5.099999904632568, filmid='tt0066164'), Row(film='Frenchie King', year=1971, votes=1054, rating=5.300000190734863, filmid='tt0067637'), Row(film='Rum Runners', year=1971, votes=469, rating=5.599999904632568, filmid='tt0066857')]",bad,False


In [345]:
spark.table("bootcamp.actors")
spark.table("bootcamp.actor_films")

DataFrame[actor: string, actorid: string, film: string, year: int, votes: int, rating: float, filmid: string]

In [10]:
from pyspark.sql.functions import current_schema

In [11]:
%%sql
current_schema()

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'current_schema'.(line 1, pos 0)

== SQL ==
current_schema()
^^^


In [ ]:


WITH streak_started AS (
    SELECT 
        actor
        , year
        , quality_class
        , is_active
        , LAG(quality_class, 1) OVER
            (PARTITION BY actor ORDER BY year) <> quality_class
            OR LAG(quality_class, 1) OVER
            (PARTITION BY actor ORDER BY year) IS NULL
            AS quality_class_did_change
        , LAG(is_active, 1) OVER
            (PARTITION BY actor ORDER BY year) <> is_active
            OR LAG(is_active, 1) OVER
            (PARTITION BY actor ORDER BY year) IS NULL
            AS is_active_did_change    
    FROM actors
),
    streak_identified AS (
        SELECT 
            actor
            , year
            , quality_class
            , is_active
            , SUM(CASE WHEN quality_class_did_change OR is_active_did_change THEN 1 ELSE 0 END)
                    OVER (PARTITION BY actor ORDER BY year) as streak_identifier
        FROM streak_started
    
    ),
     aggregated AS (
         SELECT
            actor
            , quality_class
            , is_active
            , MIN(year) AS start_date
            , MAX(year) AS end_date
         FROM streak_identified
         GROUP BY 1,2,3
    )

    SELECT actor, quality_class, is_active, start_date, end_date
    FROM aggregated
    ORDER BY 1,4,5,2,3
    ;

-- SELECT * FROM actors_history_scd


In [ ]:
- Create new PySpark jobs in `src/jobs` for these queries


In [ ]:
- Create tests in `src/tests` folder with fake input and expected output data
